In [1]:
import sys, torch
sys.path.insert(0, '.')

from functools import partial
from torch.utils.data import DataLoader

from dataset   import Vocabulary, TranslationDataset, collate_fn
from model     import Transformer
from train     import train
from inference import greedy_decode, beam_search

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using: {DEVICE}")

# ── 1. Generate reverse-string pairs ─────────────────────────────────────────
import random, string

def make_reverse_pairs(n=2000, min_len=3, max_len=12):
    src, tgt = [], []
    chars = string.ascii_lowercase        # a-z only
    for _ in range(n):
        length = random.randint(min_len, max_len)
        word   = [random.choice(chars) for _ in range(length)]
        src.append(word)
        tgt.append(list(reversed(word)))  # e.g. ['a','b','c'] → ['c','b','a']
    return src, tgt

src_sents, tgt_sents = make_reverse_pairs(n=10000)

# Quick sanity check
for s, t in zip(src_sents[:5], tgt_sents[:5]):
    print(f"  {''.join(s)}  →  {''.join(t)}")

Using: cpu
  alfjscqo  →  oqcsjfla
  nbren  →  nerbn
  onwhgkplysc  →  csylpkghwno
  mcwrmnsgv  →  vgsnmrwcm
  dalco  →  oclad


In [2]:
# Characters are the tokens, so src and tgt share the same vocab
src_vocab = Vocabulary()
tgt_vocab = Vocabulary()
src_vocab.build(src_sents)
tgt_vocab.build(tgt_sents)

print(f"Vocab size: {len(src_vocab)}")   # should be ~30 (26 letters + 4 specials)

# 80/20 train-val split
split = int(len(src_sents) * 0.8)
train_ds = TranslationDataset(src_sents[:split],  tgt_sents[:split],  src_vocab, tgt_vocab)
val_ds   = TranslationDataset(src_sents[split:],  tgt_sents[split:],  src_vocab, tgt_vocab)

pad_collate = partial(collate_fn, pad_idx=Vocabulary.PAD)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,  collate_fn=pad_collate)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False, collate_fn=pad_collate)

print(f"Train samples: {len(train_ds)} | Val samples: {len(val_ds)}")

Vocab size: 30
Train samples: 8000 | Val samples: 2000


In [ ]:
import torch, torch.nn as nn
from torch.optim import Adam
import math, time

# model 
model = Transformer(
    src_vocab_size = len(src_vocab),
    tgt_vocab_size = len(tgt_vocab),
    d_model    = 128,
    h          = 4,
    N          = 3,
    d_ff       = 256,
    dropout    = 0.1,
    pad_idx    = Vocabulary.PAD,
    tie_weights = True,
).to(DEVICE)

# ── Flat LR instead of the warmup schedule ───────────────────────────────────
optimizer = Adam(model.parameters(), lr=3e-4, betas=(0.9, 0.98), eps=1e-9)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3
)

from train import LabelSmoothingLoss, make_teacher_forcing_pair
criterion = LabelSmoothingLoss(len(tgt_vocab), pad_idx=Vocabulary.PAD, smoothing=0.1)

# ── Training loop ─────────────────────────────────────────────────────────────
def run_epoch(loader, training=True):
    model.train(training)
    total_loss, total_toks = 0.0, 0
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for batch in loader:
            src      = batch["src"].to(DEVICE)
            tgt_full = batch["tgt"].to(DEVICE)
            tgt_in, tgt_out = make_teacher_forcing_pair(tgt_full, Vocabulary.SOS, Vocabulary.EOS)

            logits = model(src, tgt_in)
            B, T, V = logits.shape
            loss = criterion(logits.reshape(B*T, V), tgt_out.reshape(B*T))

            if training:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            non_pad     = (tgt_out != Vocabulary.PAD).sum().item()
            total_loss += loss.item() * non_pad
            total_toks += non_pad

    avg = total_loss / max(total_toks, 1)
    return avg, math.exp(min(avg, 100))

print(f"{'Epoch':>6}  {'Train Loss':>10}  {'Train PPL':>10}  {'Val Loss':>10}  {'Val PPL':>10}")
print("─" * 55)

for epoch in range(1, 31):
    tr_loss, tr_ppl = run_epoch(train_loader, training=True)
    vl_loss, vl_ppl = run_epoch(val_loader,   training=False)
    scheduler.step(vl_loss)
    print(f"{epoch:>6}  {tr_loss:>10.4f}  {tr_ppl:>10.2f}  {vl_loss:>10.4f}  {vl_ppl:>10.2f}")
    if vl_ppl < 1.05:
        print(f"\n  ✓ Converged at epoch {epoch}")
        break

 Epoch  Train Loss   Train PPL    Val Loss     Val PPL
───────────────────────────────────────────────────────
     1      2.3165       10.14      1.8732        6.51
     2      1.7135        5.55      1.0383        2.82
     3      1.0933        2.98      0.2887        1.33
     4      0.6768        1.97      0.1516        1.16
     5      0.4600        1.58      0.1419        1.15
     6      0.3462        1.41      0.1366        1.15
     7      0.2623        1.30      0.1332        1.14
     8      0.2121        1.24      0.1557        1.17
     9      0.1781        1.19      0.0959        1.10
    10      0.1519        1.16      0.0968        1.10
    11      0.1347        1.14      0.0771        1.08
    12      0.1182        1.13      0.0693        1.07
    13      0.1038        1.11      0.0729        1.08
    14      0.0959        1.10      0.0708        1.07
    15      0.0851        1.09      0.0568        1.06
    16      0.0799        1.08      0.0523        1.05
    17   

In [11]:
model.eval()

test_words = ["hello", "world", "abc", "transformer", "python", "abcde"]

print("\n─── Greedy decode ───────────────────────────────")
for word in test_words:
    src = torch.tensor(
        [src_vocab.encode(list(word))], dtype=torch.long, device=DEVICE
    )
    greedy_ids = greedy_decode(model, src, Vocabulary.SOS, Vocabulary.EOS, device=DEVICE)
    greedy_out = "".join(tgt_vocab.idx2token[i] for i in greedy_ids
                         if i not in {Vocabulary.PAD, Vocabulary.SOS, Vocabulary.EOS, Vocabulary.UNK})

    correct    = word[::-1]
    status     = "✓" if greedy_out == correct else "✗"
    print(f"  {status}  {word:15s} → {greedy_out:15s}  (expected: {correct})")

print("\n─── Beam search (beam=4) ────────────────────────")
for word in test_words:
    src = torch.tensor(
        [src_vocab.encode(list(word))], dtype=torch.long, device=DEVICE
    )
    beam_ids = beam_search(model, src, Vocabulary.SOS, Vocabulary.EOS, beam_size=4, device=DEVICE)
    beam_out = "".join(tgt_vocab.idx2token[i] for i in beam_ids
                       if i not in {Vocabulary.PAD, Vocabulary.SOS, Vocabulary.EOS, Vocabulary.UNK})

    correct  = word[::-1]
    status   = "✓" if beam_out == correct else "✗"
    print(f"  {status}  {word:15s} → {beam_out:15s}  (expected: {correct})")


─── Greedy decode ───────────────────────────────
  ✓  hello           → olleh            (expected: olleh)
  ✓  world           → dlrow            (expected: dlrow)
  ✓  abc             → cba              (expected: cba)
  ✓  transformer     → remrofsnart      (expected: remrofsnart)
  ✓  python          → nohtyp           (expected: nohtyp)
  ✓  abcde           → edcba            (expected: edcba)

─── Beam search (beam=4) ────────────────────────
  ✓  hello           → olleh            (expected: olleh)
  ✓  world           → dlrow            (expected: dlrow)
  ✓  abc             → cba              (expected: cba)
  ✓  transformer     → remrofsnart      (expected: remrofsnart)
  ✓  python          → nohtyp           (expected: nohtyp)
  ✓  abcde           → edcba            (expected: edcba)
